# SA-CUT: Structure-Anchored Virtual H&E Staining

**Structure-Anchored Contrastive Unpaired Translation** for generating virtual H&E images from label-free two-photon autofluorescence (TPAF) microscopy images.

---

## Quick-start checklist
1. `Runtime → Change runtime type → T4 GPU` (or A100 if available)
2. Run **Cell 1** → verify GPU
3. Run **Cell 2** → mount Google Drive
4. Edit **Cell 4** → set your data paths
5. Run all remaining cells in order

### Expected Drive layout
```
MyDrive/SA-CUT/
├── patches/
│   ├── tpaf/          ← TPAF patches  (.npy | .png | .tif)
│   ├── he/            ← H&E  patches  (.npy | .png | .tif)
│   └── masks/         ← (optional) pre-computed nuclear masks (.npy)
├── checkpoints/       ← saved model weights (auto-created)
└── results/           ← logs + sample images (auto-created)
```

## 1 · Environment

In [ ]:
# ── GPU check ─────────────────────────────────────────────────────────────────
import subprocess, sys

gpu_info = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if gpu_info.returncode != 0:
    raise SystemExit(
        'No GPU detected.\n'
        'Go to Runtime → Change runtime type → Hardware accelerator → GPU'
    )
print(gpu_info.stdout)

import torch
print(f'PyTorch {torch.__version__} | CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Install dependencies ───────────────────────────────────────────────────────
# Colab ships with torch/torchvision; we install the remaining packages.
# openslide-tools is the system library required by openslide-python.
!apt-get install -qq openslide-tools libvips-dev
!pip install -q \
    tifffile==2024.2.12 \
    monai==1.3.0 \
    openslide-python==4.0.0 \
    pytorch-fid==0.3.0 \
    wandb==0.16.4 \
    pyyaml==6.0.1 \
    scikit-image==0.22.0 \
    cellpose

print('\nAll packages installed.')

## 2 · Mount Google Drive & clone repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os
DRIVE_ROOT = '/content/drive/MyDrive/SA-CUT'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print(f'Drive root: {DRIVE_ROOT}')

In [ ]:
# ── Clone / update SA-CUT repository ──────────────────────────────────────────
import os

REPO_DIR = '/content/SA-CUT'
REPO_URL = 'https://github.com/YOUR_USERNAME/SA-CUT.git'  # ← edit if needed
BRANCH   = 'main'                                          # ← edit if needed

if os.path.exists(os.path.join(REPO_DIR, '.git')):
    print('Repo already cloned — pulling latest changes.')
    !git -C {REPO_DIR} pull origin {BRANCH} --ff-only
else:
    print('Cloning repository...')
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}

# Alternatively, copy from Drive if you uploaded the code there:
# !cp -r /content/drive/MyDrive/SA-CUT-code /content/SA-CUT

os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')
!git log --oneline -5

## 3 · Data paths & experiment config

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  EDIT THIS CELL — all paths and key hyper-parameters live here  ║
# ╚══════════════════════════════════════════════════════════════════╝
import os

DRIVE_ROOT = '/content/drive/MyDrive/SA-CUT'

# ── Data directories ──────────────────────────────────────────────────────────
TPAF_DIR = os.path.join(DRIVE_ROOT, 'patches/tpaf')
HE_DIR   = os.path.join(DRIVE_ROOT, 'patches/he')
MASK_DIR = os.path.join(DRIVE_ROOT, 'patches/masks')

# ── Cellpose-SAM checkpoint ────────────────────────────────────────────────────
# Cellpose-saved models have no file extension — this is normal.
# model_type must match the checkpoint architecture:
#   'cpsam'  → Cellpose-SAM (SAM backbone; filenames start with 'cpsam_*')
#   'cyto3'  → classic Cellpose U-Net  (incompatible with cpsam checkpoints)
CELLPOSE_CKPT               = '/content/drive/MyDrive/SA-CUT/cellpose-SAM_checkpoints/cpsam_20260228_gray'
CELLPOSE_MODEL_TYPE         = 'cpsam'
CELLPOSE_DIAMETER           = 30    # expected nucleus diameter in pixels; 0 = auto
CELLPOSE_FLOW_THRESHOLD     = 0.4
CELLPOSE_CELLPROB_THRESHOLD = 0.0

# ── Output directories ────────────────────────────────────────────────────────
CKPT_DIR    = os.path.join(DRIVE_ROOT, 'checkpoints')
LOG_DIR     = os.path.join(DRIVE_ROOT, 'results/logs')
RESUME_CKPT = ''

# ── Experiment name ───────────────────────────────────────────────────────────
from datetime import datetime
EXP_NAME = f'sa_cut_{datetime.now().strftime("%Y%m%d")}_colab'

# ── Key hyper-parameters ──────────────────────────────────────────────────────
N_EPOCHS       = 200
N_EPOCHS_DECAY = 200
BATCH_SIZE     = 1
LR_G           = 2e-4
LR_D           = 2e-4
LAMBDA_STRUCT  = 5.0
STRUCT_WARMUP  = 10
MASK_MODE      = 'precomputed' if (MASK_DIR and os.path.isdir(MASK_DIR)) else 'cellpose_sam'
USE_AMP        = True

# ── W&B (optional) ────────────────────────────────────────────────────────────
USE_WANDB     = False
WANDB_PROJECT = 'SA-CUT'

# ── Path validation ───────────────────────────────────────────────────────────
for label, d in [('TPAF', TPAF_DIR), ('H&E', HE_DIR)]:
    if not os.path.isdir(d):
        print(f'[WARNING] {label} dir not found: {d}')
    else:
        n = sum(1 for f in __import__('pathlib').Path(d).rglob('*') if f.is_file())
        print(f'  {label:4s}: {d}  ({n} files)')

ckpt_ok = os.path.isfile(CELLPOSE_CKPT)
print(f'\n  Cellpose ckpt : {CELLPOSE_CKPT}')
print(f'                  {"✓ found" if ckpt_ok else "✗ NOT FOUND — check path"}')
print(f'  model_type    : {CELLPOSE_MODEL_TYPE}')
print(f'  Mask mode     : {MASK_MODE}')

In [ ]:
# ── Write Colab-specific experiment YAML ──────────────────────────────────────
import yaml, os

colab_cfg = {
    'experiment': {
        'name':          EXP_NAME,
        'log_dir':       LOG_DIR,
        'checkpoint_dir': CKPT_DIR,
        'save_freq':     10,
        'log_freq':      50,
        'vis_freq':      200,
        'use_wandb':     USE_WANDB,
        'use_tensorboard': True,
    },
    'data': {
        'tpaf_dir':   TPAF_DIR,
        'he_dir':     HE_DIR,
        'patch_size': 256,
        'num_workers': 2,
        'augment':    True,
    },
    'mask_provider': {
        'mode':     MASK_MODE,
        'mask_dir': MASK_DIR,
    },
    'losses': {
        'lambda_struct':       LAMBDA_STRUCT,
        'struct_warmup_epochs': STRUCT_WARMUP,
        'struct_rampup_epochs': 5,
    },
    'training': {
        'batch_size':    BATCH_SIZE,
        'lr_G':          LR_G,
        'lr_D':          LR_D,
        'n_epochs':      N_EPOCHS,
        'n_epochs_decay': N_EPOCHS_DECAY,
        'use_amp':       USE_AMP,
        'seed':          42,
    },
}

COLAB_CFG_PATH = '/content/SA-CUT/configs/colab_experiment.yaml'
with open(COLAB_CFG_PATH, 'w') as f:
    yaml.dump(colab_cfg, f, default_flow_style=False)

print(f'Config written → {COLAB_CFG_PATH}')
print(yaml.dump(colab_cfg, default_flow_style=False))

## 4 · (Optional) W&B login

In [ ]:
if USE_WANDB:
    import wandb
    wandb.login()   # prompts for API key; paste from wandb.ai/authorize
    print('W&B authenticated.')
else:
    print('W&B disabled — set USE_WANDB=True in Cell 4 to enable.')

## 5 · (Optional) Extract patches from WSI

In [ ]:
# Skip this cell if you already have pre-extracted patches in TPAF_DIR / HE_DIR.
#
# Example usage:
#   TPAF_WSI_DIR — directory of .tif TPAF whole-slide images
#   HE_WSI_DIR   — directory of .svs / .ndpi H&E whole-slide images
#
# The extractor saves float32 .npy patches with percentile normalisation.

RUN_EXTRACTION = False  # ← set True to run

TPAF_WSI_DIR  = os.path.join(DRIVE_ROOT, 'wsi/tpaf')
HE_WSI_DIR    = os.path.join(DRIVE_ROOT, 'wsi/he')
PATCH_SIZE    = 256
STRIDE        = 256   # non-overlapping patches
BG_THRESHOLD  = 0.95  # discard patches with > 95% background

if RUN_EXTRACTION:
    import sys; sys.path.insert(0, '/content/SA-CUT')
    from data.patch_extractor import PatchExtractor

    for domain, wsi_dir, out_dir in [
        ('tpaf', TPAF_WSI_DIR, TPAF_DIR),
        ('he',   HE_WSI_DIR,   HE_DIR),
    ]:
        os.makedirs(out_dir, exist_ok=True)
        extractor = PatchExtractor(
            wsi_dir=wsi_dir,
            out_dir=out_dir,
            domain=domain,
            patch_size=PATCH_SIZE,
            stride=STRIDE,
            bg_threshold=BG_THRESHOLD,
        )
        n = extractor.run()
        print(f'[{domain}] extracted {n} patches → {out_dir}')
else:
    print('Extraction skipped. Using existing patches in:')
    print(f'  TPAF: {TPAF_DIR}')
    print(f'  H&E:  {HE_DIR}')

## 6 · Cellpose-SAM nuclear segmentation → precompute masks

Run this section **once** before training to generate `masks/` from your TPAF patches.  
Pre-saving masks eliminates on-the-fly Cellpose inference during the training loop (~3–5× speedup per iteration).

- **Recommended path**: upload your fine-tuned `cellpose_sam_tpaf.pth` to `Drive/SA-CUT/checkpoints/` then run this cell.
- **No fine-tuned checkpoint?** Set `CELLPOSE_CKPT = ''` — Cellpose will use the built-in `cyto3` weights (pre-trained on general cell images, less accurate on TPAF but functional).
- **Already have masks?** Skip to the next section.

In [ ]:
# ── Section 6 · Cellpose-SAM nuclear segmentation ─────────────────────────────
#
# Loads the fine-tuned Cellpose-SAM checkpoint and runs nuclei segmentation
# on every TPAF patch in TPAF_DIR.  Saves one binary float32 .npy mask per
# patch to MASK_DIR (shape [1, H, W], values {0.0, 1.0}).
#
# Also writes two QC artefacts to MASK_DIR:
#   qc_summary.csv   — per-patch: num_nuclei, mask_coverage_ratio, mean_area
#   qc_montage.png   — 4×4 grid of TPAF patches with red mask overlays
#
# ── Paths (edit here if different from Cell 4) ────────────────────────────────
import os, sys, csv, torch
from pathlib import Path
from IPython.display import Image as IPImage, display

sys.path.insert(0, '/content/SA-CUT')

TPAF_INPUT_DIR  = '/content/drive/MyDrive/SA-CUT/patches/tpaf'
MASK_OUTPUT_DIR = '/content/drive/MyDrive/SA-CUT/patches/masks'
CKPT_PATH       = '/content/drive/MyDrive/SA-CUT/cellpose-SAM_checkpoints/cpsam_20260228_gray'
MODEL_TYPE      = 'cpsam'   # must match checkpoint architecture
DIAMETER        = 30        # nucleus diameter in pixels; set None for auto-estimate
FLOW_THRESH     = 0.4
CELLPROB_THRESH = 0.0

# ── Pre-flight checks ─────────────────────────────────────────────────────────
assert os.path.isdir(TPAF_INPUT_DIR),  f'TPAF dir not found: {TPAF_INPUT_DIR}'
assert os.path.isfile(CKPT_PATH),      f'Checkpoint not found: {CKPT_PATH}'

tpaf_files = sorted(
    p for p in Path(TPAF_INPUT_DIR).rglob('*')
    if p.is_file() and p.suffix.lower() in {'.tif', '.tiff', '.png', '.npy'}
)
assert tpaf_files, f'No TPAF patches found in {TPAF_INPUT_DIR}'

os.makedirs(MASK_OUTPUT_DIR, exist_ok=True)
use_gpu = torch.cuda.is_available()

print(f'Checkpoint  : {CKPT_PATH}')
print(f'Model type  : {MODEL_TYPE}')
print(f'TPAF patches: {len(tpaf_files)}  in  {TPAF_INPUT_DIR}')
print(f'Mask output : {MASK_OUTPUT_DIR}')
print(f'GPU         : {use_gpu}  |  diameter: {DIAMETER}')
print()

# ── Load Cellpose-SAM (frozen) ────────────────────────────────────────────────
from cellpose import models as cp_models

model = cp_models.CellposeModel(
    pretrained_model=CKPT_PATH,
    model_type=MODEL_TYPE,
    gpu=use_gpu,
)
# Freeze — this script never back-props through Cellpose
for attr in ('net', 'net2'):
    net = getattr(model, attr, None)
    if net is not None and hasattr(net, 'parameters'):
        net.eval()
        for p in net.parameters():
            p.requires_grad_(False)
print('Cellpose-SAM loaded and frozen.')

# ── Batch inference ───────────────────────────────────────────────────────────
from scripts.precompute_masks import _load_tpaf_patch, _patch_qc_stats, _save_qc_csv, _save_qc_montage
from tqdm.auto import tqdm

qc_rows      = []
montage_data = []   # (tpaf_hw, binary_hw) tuples for the QC montage

for patch_path in tqdm(tpaf_files, desc='Segmenting nuclei', unit='patch'):
    try:
        tpaf_hw = _load_tpaf_patch(patch_path)   # float32 (H, W) in [0, 1]
    except Exception as e:
        print(f'  [skip] {patch_path.name}: {e}')
        continue

    # Cellpose expects [0, 255] range
    img_cp = (tpaf_hw * 255.0).astype('float32')

    with torch.no_grad():
        masks_inst, flows, _ = model.eval(
            img_cp,
            diameter        = DIAMETER,
            channels        = [0, 0],   # single-channel grayscale input
            flow_threshold  = FLOW_THRESH,
            cellprob_threshold = CELLPROB_THRESH,
            normalize       = True,
        )

    # Instance label map → binary float32 mask, shape (1, H, W)
    binary_hw  = (masks_inst > 0).astype('float32')
    mask_out   = binary_hw[None]                         # (1, H, W)

    # Save mask — same stem as TPAF file, always .npy
    out_path = Path(MASK_OUTPUT_DIR) / f'{patch_path.stem}.npy'
    import numpy as np
    np.save(str(out_path), mask_out)

    qc_rows.append(_patch_qc_stats(binary_hw, masks_inst, patch_path.name))
    montage_data.append((tpaf_hw, binary_hw))

print(f'\nDone — {len(qc_rows)}/{len(tpaf_files)} patches segmented.')
print(f'Masks saved to: {MASK_OUTPUT_DIR}')

# ── QC summary ────────────────────────────────────────────────────────────────
_save_qc_csv(qc_rows, Path(MASK_OUTPUT_DIR))

coverages  = [r['mask_coverage_ratio']    for r in qc_rows]
nuclei     = [r['num_nuclei_detected']    for r in qc_rows]
mean_areas = [r['mean_nucleus_area']      for r in qc_rows]
n = len(qc_rows)

print(f'\nQC summary ({n} patches):')
print(f'  Coverage ratio    mean={sum(coverages)/n:.3f}  '
      f'min={min(coverages):.3f}  max={max(coverages):.3f}')
print(f'  Nuclei detected   mean={sum(nuclei)/n:.1f}  '
      f'min={min(nuclei)}  max={max(nuclei)}')
print(f'  Mean nucleus area {sum(mean_areas)/n:.1f} px²')

# ── QC montage ────────────────────────────────────────────────────────────────
import random
random.shuffle(montage_data)
_save_qc_montage(montage_data[:16], Path(MASK_OUTPUT_DIR))

montage_png = Path(MASK_OUTPUT_DIR) / 'qc_montage.png'
if montage_png.exists():
    print('\nQC montage — TPAF + nuclei mask overlay (red):')
    display(IPImage(filename=str(montage_png), width=900))

# Sync MASK_MODE so downstream training cells pick up the new masks
MASK_MODE = 'precomputed'
print(f'\nMASK_MODE → "{MASK_MODE}"  (ready for training)')

## 6 · Dataset sanity check

In [ ]:
import sys; sys.path.insert(0, '/content/SA-CUT')
import torch
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from data.dataset import UnpairedTPAFDataset
from data.transforms import UnpairedTransform

transform = UnpairedTransform(
    p_hflip=0.5, p_vflip=0.5, p_rot90=0.5,
    he_color_jitter=True,
)
ds = UnpairedTPAFDataset(
    tpaf_dir  = TPAF_DIR,
    he_dir    = HE_DIR,
    mask_dir  = MASK_DIR or None,
    patch_size= 256,
    transform = transform,
)
print(ds)
print(f'Dataset length: {len(ds)}')

# Load one sample and check shapes / ranges
sample = ds[0]
for k, v in sample.items():
    if isinstance(v, torch.Tensor):
        print(f'  {k:12s}: shape={tuple(v.shape)}  dtype={v.dtype}  '
              f'min={v.min():.3f}  max={v.max():.3f}')

# Visualise
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
tpaf_np = sample['tpaf'][0].numpy()
he_np   = (sample['he'].permute(1,2,0).numpy() + 1) / 2.0   # [-1,1] → [0,1]
mask_np = sample['mask'][0].numpy()

axes[0].imshow(tpaf_np, cmap='gray'); axes[0].set_title('TPAF (src)')
axes[1].imshow(he_np.clip(0,1));      axes[1].set_title('H&E (tgt) [0,1] repr.')
axes[2].imshow(mask_np, cmap='hot');  axes[2].set_title('Nuclear mask')
for ax in axes: ax.axis('off')
plt.tight_layout()
plt.savefig('/tmp/sanity_check.png', dpi=100)
plt.show()
print('Sanity check passed ✓')

## 7 · TensorBoard

In [ ]:
import os
os.makedirs(LOG_DIR, exist_ok=True)

%load_ext tensorboard
%tensorboard --logdir {LOG_DIR} --port 6006

## 8 · Train

In [ ]:
import os
for d in [CKPT_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)
    print(f'Created: {d}')

In [ ]:
# ── Launch training ────────────────────────────────────────────────────────────
# This cell runs the full training loop inside the notebook process.
# Training is checkpointed to Drive every `save_freq` epochs, so it
# survives Colab disconnects — just re-run with RESUME_CKPT set.
import sys, os
sys.path.insert(0, '/content/SA-CUT')
os.chdir('/content/SA-CUT')

import random
import numpy as np
import torch

from configs.config_utils import load_config
from trainers.sa_cut_trainer import SACUTTrainer

# ── Seed ──────────────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Config ────────────────────────────────────────────────────────────────────
cfg = load_config(
    config_path = COLAB_CFG_PATH,
    base_path   = '/content/SA-CUT/configs/default.yaml',
)

# ── Build trainer ─────────────────────────────────────────────────────────────
trainer = SACUTTrainer(cfg)

# ── Resume (optional) ─────────────────────────────────────────────────────────
if RESUME_CKPT and os.path.isfile(RESUME_CKPT):
    print(f'Resuming from: {RESUME_CKPT}')
    trainer.load_checkpoint(RESUME_CKPT)

# ── Train ─────────────────────────────────────────────────────────────────────
print(f'\nStarting training: {EXP_NAME}')
print(f'  Epochs: {N_EPOCHS} + {N_EPOCHS_DECAY} decay = {N_EPOCHS + N_EPOCHS_DECAY} total')
print(f'  Checkpoints → {CKPT_DIR}')
print(f'  Logs        → {LOG_DIR}\n')

try:
    trainer.train()
except KeyboardInterrupt:
    print('\nInterrupted — saving emergency checkpoint...')
    trainer.save_checkpoint(trainer.current_epoch, tag='emergency')
    print(f'Saved to {CKPT_DIR}/emergency.pth')

print('\nTraining complete.')

## 9 · Inference on new TPAF images

In [ ]:
# ── Inference ─────────────────────────────────────────────────────────────────
import sys, os, glob
sys.path.insert(0, '/content/SA-CUT')

import torch
import numpy as np
from PIL import Image
from pathlib import Path
import matplotlib.pyplot as plt

from models.generator import ResNetGenerator
from models.networks import init_weights

# ── Settings ──────────────────────────────────────────────────────────────────
INFER_CKPT   = os.path.join(CKPT_DIR, 'latest.pth')   # or 'epoch_200.pth'
INFER_DIR    = TPAF_DIR                                 # directory of TPAF patches to translate
INFER_OUT    = os.path.join(DRIVE_ROOT, 'results/inference')
INFER_LIMIT  = 16   # number of patches to translate (set None for all)
os.makedirs(INFER_OUT, exist_ok=True)

# ── Load checkpoint ───────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt   = torch.load(INFER_CKPT, map_location=device)

G = ResNetGenerator(
    input_nc       = ckpt['config']['generator']['input_nc'],
    output_nc      = ckpt['config']['generator']['output_nc'],
    ngf            = ckpt['config']['generator']['ngf'],
    n_resnet_blocks= ckpt['config']['generator']['n_resnet_blocks'],
    norm_type      = ckpt['config']['generator']['norm_type'],
    use_dropout    = ckpt['config']['generator']['use_dropout'],
).to(device)
G.load_state_dict(ckpt['G_state_dict'])
G.eval()
print(f'Generator loaded from epoch {ckpt["epoch"]}.')

# ── Helpers ───────────────────────────────────────────────────────────────────
def load_tpaf_patch(path: str) -> torch.Tensor:
    """Load a TPAF patch (any supported format) → (1,1,H,W) float32 [0,1]."""
    p = Path(path)
    if p.suffix == '.npy':
        arr = np.load(p).astype(np.float32)
    elif p.suffix == '.png':
        arr = np.array(Image.open(p).convert('L'), dtype=np.float32)
        lo, hi = np.percentile(arr, [1, 99])
        arr = np.clip((arr - lo) / (hi - lo + 1e-6), 0, 1)
    else:  # .tif / .tiff
        import tifffile
        arr = tifffile.imread(str(p)).astype(np.float32)
        lo, hi = np.percentile(arr, [1, 99])
        arr = np.clip((arr - lo) / (hi - lo + 1e-6), 0, 1)
    if arr.ndim == 2:
        arr = arr[None]
    return torch.from_numpy(arr).unsqueeze(0).to(device)  # (1,1,H,W)

def dummy_mask(tpaf: torch.Tensor) -> torch.Tensor:
    """Return a zero mask matching the spatial size of tpaf."""
    return torch.zeros_like(tpaf)

# ── Run inference ─────────────────────────────────────────────────────────────
patch_files = sorted(glob.glob(os.path.join(INFER_DIR, '**/*.*'), recursive=True))
patch_files = [f for f in patch_files if Path(f).suffix in ('.npy','.png','.tif','.tiff')]
if INFER_LIMIT:
    patch_files = patch_files[:INFER_LIMIT]

print(f'Translating {len(patch_files)} patches...')
saved = []
with torch.no_grad():
    for fpath in patch_files:
        tpaf  = load_tpaf_patch(fpath)
        mask  = dummy_mask(tpaf)
        inp   = torch.cat([tpaf, mask], dim=1)   # (1,2,H,W)
        fake_he = G(inp)                           # (1,3,H,W) in [-1,1]
        fake_he = (fake_he.clamp(-1, 1) + 1) / 2  # → [0,1]
        out_arr = (fake_he[0].permute(1,2,0).cpu().numpy() * 255).astype(np.uint8)
        out_path = os.path.join(INFER_OUT, Path(fpath).stem + '_he.png')
        Image.fromarray(out_arr).save(out_path)
        saved.append((fpath, out_path, fake_he[0].cpu()))

print(f'Saved {len(saved)} images → {INFER_OUT}')

# ── Visualise first 8 ─────────────────────────────────────────────────────────
n_vis = min(8, len(saved))
fig, axes = plt.subplots(2, n_vis, figsize=(3*n_vis, 6))
for i, (src, _, he_t) in enumerate(saved[:n_vis]):
    tpaf_arr = load_tpaf_patch(src)[0, 0].cpu().numpy()
    he_arr   = (he_t.permute(1,2,0).numpy()).clip(0,1)
    axes[0, i].imshow(tpaf_arr, cmap='gray'); axes[0, i].set_title('TPAF')
    axes[1, i].imshow(he_arr);               axes[1, i].set_title('Virtual H&E')
    axes[0, i].axis('off'); axes[1, i].axis('off')
plt.suptitle('SA-CUT: TPAF → Virtual H&E', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(INFER_OUT, 'grid.png'), dpi=120)
plt.show()

## 10 · Evaluation

In [ ]:
# ── FID score ─────────────────────────────────────────────────────────────────
# Requires INFER_OUT to be populated (run Cell 9 first).

import subprocess

FID_REAL_DIR = HE_DIR        # real H&E patches as reference
FID_FAKE_DIR = INFER_OUT     # generated virtual H&E patches
FID_BATCH    = 50

result = subprocess.run(
    ['python', '-m', 'pytorch_fid',
     '--batch-size', str(FID_BATCH),
     FID_REAL_DIR, FID_FAKE_DIR],
    capture_output=True, text=True, cwd='/content/SA-CUT'
)
print(result.stdout)
if result.returncode != 0:
    print('[stderr]', result.stderr)

In [ ]:
# ── Structure consistency (Mask IoU / F1) ─────────────────────────────────────
# Compares the input nuclear mask to a mask extracted from the generated H&E.

import sys
sys.path.insert(0, '/content/SA-CUT')

from evaluation.structure_consistency import evaluate_structure_consistency

metrics = evaluate_structure_consistency(
    generated_dir = INFER_OUT,
    mask_dir      = MASK_DIR or None,
    iou_threshold = 0.5,
)

print('Structure Consistency Metrics')
print('──────────────────────────────')
for k, v in metrics.items():
    print(f'  {k:25s}: {v:.4f}')

## 11 · Save / download artefacts

In [ ]:
# All checkpoints and results are already written to Google Drive.
# Use this cell to additionally zip and download a specific artefact.

import os, shutil
from google.colab import files

# Example: download the latest checkpoint
latest_ckpt = os.path.join(CKPT_DIR, 'latest.pth')
if os.path.isfile(latest_ckpt):
    files.download(latest_ckpt)
    print(f'Downloading: {latest_ckpt}')
else:
    print(f'No checkpoint found at {latest_ckpt}.')

# Example: download the inference grid
grid_path = os.path.join(INFER_OUT, 'grid.png')
if os.path.isfile(grid_path):
    files.download(grid_path)